In [2]:

import pandas as pd

df = pd.read_csv("cybersecurity_ir_10_documents.csv")

print("Number of documents:", len(df))
display(df[["Document_ID", "Title"]])

assert len(df) >= 10, "The assignment requires at least 10 documents."
assert df["Text"].notna().all(), "Some document text is missing."

print("\nExample original document:")
print(df.iloc[0]["Text"])


Number of documents: 10


,Document_ID,Title
0,D1,Phishing Email Attacks
1,D2,Malware and Antivirus Protection
2,D3,Ransomware Incident Response
3,D4,Network Firewalls and Intrusion Detection
4,D5,Encryption and Data Privacy
5,D6,Password Managers and Strong Authentication
6,D7,Cloud Security and Access Control
7,D8,Public Wi-Fi and Safe Browsing
8,D9,Data Breaches and Security Awareness
9,D10,Multi-Factor Authentication for Online Accounts



Example original document:
Phishing emails trick people into revealing passwords, bank details, and private information. Attackers imitate trusted companies and create urgent messages with malicious links. Users should inspect sender addresses, avoid unexpected attachments, and report suspicious email to the security team.


In [3]:

!pip -q install nltk scikit-learn pandas


In [4]:

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    words = [word for word in words if word not in stop_words]
    return [stemmer.stem(word) for word in words]

df["Processed_Tokens"] = df["Text"].apply(preprocess)
df["Processed_Text"] = df["Processed_Tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print("Original text:\n", df.iloc[0]["Text"])
print("\nPreprocessed text:\n", df.iloc[0]["Processed_Text"])


Original text:
 Phishing emails trick people into revealing passwords, bank details, and private information. Attackers imitate trusted companies and create urgent messages with malicious links. Users should inspect sender addresses, avoid unexpected attachments, and report suspicious email to the security team.

Preprocessed text:
 phish email trick peopl reveal password bank detail privat inform attack imit trust compani creat urgent messag malici link user inspect sender address avoid unexpect attach report suspici email secur team


In [5]:

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Use the same preprocessing function for all documents
vectorizer = CountVectorizer(
    analyzer=preprocess,
    min_df=1
)

tf_matrix = vectorizer.fit_transform(df["Text"])
terms = vectorizer.get_feature_names_out()

# Term frequency (TF)
tf_df = pd.DataFrame(
    tf_matrix.toarray(),
    index=df["Document_ID"],
    columns=terms
)

# Document frequency (DF)
df_values = (tf_matrix.toarray() > 0).sum(axis=0)

# Inverse document frequency (IDF), base 10
N = len(df)
idf_values = np.log10(N / df_values)

idf_df = pd.DataFrame({
    "Term": terms,
    "DF": df_values,
    "IDF": idf_values
}).sort_values("IDF", ascending=False)

# TF-IDF = raw TF * IDF
tfidf_values = tf_matrix.toarray() * idf_values

tfidf_df = pd.DataFrame(
    tfidf_values,
    index=df["Document_ID"],
    columns=terms
)

print("TF matrix shape:", tf_df.shape)
print("\nSample TF values:")
display(tf_df.iloc[:, :12])

print("\nSample DF and IDF values:")
display(idf_df.head(20))

print("\nSample TF-IDF values:")
display(tfidf_df.iloc[:, :12].round(4))


TF matrix shape: (10, 196)

Sample TF values:


,access,accord,account,across,activ,add,address,administr,affect,allow,alon,also
Document_ID,,,,,,,,,,,,
D1,0,0,0,0,0,0,1,0,0,0,0,0
D2,0,0,0,0,0,0,0,0,0,0,0,0
D3,1,0,0,0,0,0,0,1,1,0,0,0
D4,0,1,0,0,1,0,0,0,0,0,0,0
D5,1,0,0,1,0,0,0,0,0,0,0,1
D6,1,0,1,0,0,1,0,0,0,1,0,0
D7,1,0,0,0,0,0,0,0,0,0,0,0
D8,0,0,0,0,0,1,0,0,0,0,0,0
D9,2,0,0,0,0,0,0,0,0,0,0,0



Sample DF and IDF values:


,Term,DF,IDF
1,accord,1,1.0
3,across,1,1.0
6,address,1,1.0
4,activ,1,1.0
66,grant,1,1.0
65,follow,1,1.0
9,allow,1,1.0
8,affect,1,1.0
10,alon,1,1.0
11,also,1,1.0



Sample TF-IDF values:


,access,accord,account,across,activ,add,address,administr,affect,allow,alon,also
Document_ID,,,,,,,,,,,,
D1,0.0000,0.0,0.0000,0.0,0.0,0.000,1.0,0.000,0.0,0.0,0.0,0.0
D2,0.0000,0.0,0.0000,0.0,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.0
D3,0.2218,0.0,0.0000,0.0,0.0,0.000,0.0,0.699,1.0,0.0,0.0,0.0
D4,0.0000,1.0,0.0000,0.0,1.0,0.000,0.0,0.000,0.0,0.0,0.0,0.0
D5,0.2218,0.0,0.0000,1.0,0.0,0.000,0.0,0.000,0.0,0.0,0.0,1.0
D6,0.2218,0.0,0.6990,0.0,0.0,0.699,0.0,0.000,0.0,1.0,0.0,0.0
D7,0.2218,0.0,0.0000,0.0,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.0
D8,0.0000,0.0,0.0000,0.0,0.0,0.699,0.0,0.000,0.0,0.0,0.0,0.0
D9,0.4437,0.0,0.0000,0.0,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.0


In [6]:

term = "phish"

if term not in terms:
    print("Term not found; inspect idf_df for a term to use.")
else:
    j = list(terms).index(term)
    doc_id = "D1"
    i = list(df["Document_ID"]).index(doc_id)

    tf = int(tf_matrix[i, j])
    df_term = int(df_values[j])
    idf = float(idf_values[j])
    tfidf = tf * idf

    print("Manual calculation")
    print("Document:", doc_id)
    print("Term:", term)
    print("TF =", tf)
    print("DF =", df_term)
    print("IDF = log10(10 /", df_term, ") =", round(idf, 6))
    print("TF-IDF =", tf, "*", round(idf, 6), "=", round(tfidf, 6))

    print("\nVerification against Python matrices:")
    print("TF matches:", tf == int(tf_matrix[i, j]))
    print("TF-IDF matches:",
          np.isclose(tfidf, tfidf_values[i, j]))


Manual calculation
Document: D1
Term: phish
TF = 1
DF = 2
IDF = log10(10 / 2 ) = 0.69897
TF-IDF = 1 * 0.69897 = 0.69897

Verification against Python matrices:
TF matches: True
TF-IDF matches: True


In [7]:

from sklearn.metrics.pairwise import cosine_similarity

queries = [
    "phishing email security",
    "network firewall protection",
    "password encryption privacy"
]

def query_to_tfidf(query):
    # Transform with the same analyzer and vocabulary
    tokens = preprocess(query)
    counts = {}

    for token in tokens:
        if token in vectorizer.vocabulary_:
            j = vectorizer.vocabulary_[token]
            counts[j] = counts.get(j, 0) + 1

    query_vector = np.zeros(len(terms), dtype=float)

    for j, count in counts.items():
        query_vector[j] = count * idf_values[j]

    return query_vector.reshape(1, -1)

ranked_results = []

for query in queries:
    q_vector = query_to_tfidf(query)

    # Cosine similarity between query and each document
    scores = cosine_similarity(q_vector, tfidf_values)[0]

    ranking = np.argsort(scores)[::-1]

    for rank, idx in enumerate(ranking, start=1):
        ranked_results.append({
            "Query": query,
            "Rank": rank,
            "Document_ID": df.iloc[idx]["Document_ID"],
            "Title": df.iloc[idx]["Title"],
            "Cosine_Similarity": round(float(scores[idx]), 4)
        })

results_df = pd.DataFrame(ranked_results)

for query in queries:
    print("\nQUERY:", query)
    display(
        results_df[results_df["Query"] == query]
        .head(5)
        .reset_index(drop=True)
    )



QUERY: phishing email security


,Query,Rank,Document_ID,Title,Cosine_Similarity
0,phishing email security,1,D1,Phishing Email Attacks,0.3203
1,phishing email security,2,D9,Data Breaches and Security Awareness,0.1122
2,phishing email security,3,D10,Multi-Factor Authentication for Online Accounts,0.1092
3,phishing email security,4,D7,Cloud Security and Access Control,0.0042
4,phishing email security,5,D6,Password Managers and Strong Authentication,0.0021



QUERY: network firewall protection


,Query,Rank,Document_ID,Title,Cosine_Similarity
0,network firewall protection,1,D4,Network Firewalls and Intrusion Detection,0.5067
1,network firewall protection,2,D8,Public Wi-Fi and Safe Browsing,0.1456
2,network firewall protection,3,D5,Encryption and Data Privacy,0.0693
3,network firewall protection,4,D10,Multi-Factor Authentication for Online Accounts,0.0169
4,network firewall protection,5,D2,Malware and Antivirus Protection,0.0159



QUERY: password encryption privacy


,Query,Rank,Document_ID,Title,Cosine_Similarity
0,password encryption privacy,1,D5,Encryption and Data Privacy,0.2873
1,password encryption privacy,2,D6,Password Managers and Strong Authentication,0.1184
2,password encryption privacy,3,D10,Multi-Factor Authentication for Online Accounts,0.0577
3,password encryption privacy,4,D7,Cloud Security and Access Control,0.0511
4,password encryption privacy,5,D3,Ransomware Incident Response,0.0484


In [8]:

absent_query = "quantum teleportation"

q_vector = query_to_tfidf(absent_query)
print("Query:", absent_query)
print("Preprocessed terms:", preprocess(absent_query))
print("Non-zero query weights:", np.count_nonzero(q_vector))

if np.count_nonzero(q_vector) == 0:
    print("Result: none of the query terms are in the indexed vocabulary.")
    print("All documents will have zero similarity for this query.")
else:
    scores = cosine_similarity(q_vector, tfidf_values)[0]
    print("Ranking scores:", np.round(scores, 4))


Query: quantum teleportation
Preprocessed terms: ['quantum', 'teleport']
Non-zero query weights: 0
Result: none of the query terms are in the indexed vocabulary.
All documents will have zero similarity for this query.


In [9]:

from pathlib import Path

output_dir = Path("task1_outputs")
output_dir.mkdir(exist_ok=True)

df[["Document_ID", "Title", "Text", "Processed_Text"]].to_csv(
    output_dir / "document_collection_preprocessed.csv", index=False
)

tf_df.to_csv(output_dir / "tf_matrix.csv")
idf_df.to_csv(output_dir / "df_idf_table.csv", index=False)
tfidf_df.to_csv(output_dir / "tfidf_matrix.csv")
results_df.to_csv(output_dir / "ranked_retrieval_results.csv", index=False)

print("Saved files:")
for file in sorted(output_dir.iterdir()):
    print(file)


Saved files:
task1_outputs/df_idf_table.csv
task1_outputs/document_collection_preprocessed.csv
task1_outputs/ranked_retrieval_results.csv
task1_outputs/tf_matrix.csv
task1_outputs/tfidf_matrix.csv


In [10]:

import shutil

zip_path = shutil.make_archive(
    "Task1_Cybersecurity_Results", "zip", output_dir
)

print("Created:", zip_path)

from google.colab import files
files.download(zip_path)


Created: /content/Task1_Cybersecurity_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
import pandas as pd

df = pd.read_csv("/content/cybersecurity_ir_10_documents.csv")
print("Number of documents:", len(df))

Number of documents: 10


In [12]:

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# Reuse tfidf_values, df, and query_to_tfidf
# from your earlier TF-IDF code.

queries = [
    "phishing email security",
    "network firewall protection",
    "password encryption privacy"
]

all_rankings = []

for query in queries:
    query_vector = query_to_tfidf(query)

    if np.count_nonzero(query_vector) == 0:
        scores = np.zeros(len(df))
    else:
        scores = cosine_similarity(
            query_vector, tfidf_values
        )[0]

    ranking = np.argsort(scores)[::-1]

    print("\nQuery:", query)
    for rank, index in enumerate(ranking, start=1):
        doc_id = df.iloc[index]["Document_ID"]
        title = df.iloc[index]["Title"]
        score = scores[index]

        all_rankings.append({
            "Query": query,
            "Rank": rank,
            "Document_ID": doc_id,
            "Title": title,
            "Cosine_Similarity": round(float(score), 4)
        })

        print(
            f"{rank}. {doc_id} - {title} "
            f"| Similarity: {score:.4f}"
        )

# Save the results
results_df = pd.DataFrame(all_rankings)
output_folder = Path("03_Results")
output_folder.mkdir(exist_ok=True)

results_df.to_csv(
    output_folder / "ranked_retrieval_results.csv",
    index=False
)

print("\nSaved ranked_retrieval_results.csv")



Query: phishing email security
1. D1 - Phishing Email Attacks | Similarity: 0.3203
2. D9 - Data Breaches and Security Awareness | Similarity: 0.1122
3. D10 - Multi-Factor Authentication for Online Accounts | Similarity: 0.1092
4. D7 - Cloud Security and Access Control | Similarity: 0.0042
5. D6 - Password Managers and Strong Authentication | Similarity: 0.0021
6. D5 - Encryption and Data Privacy | Similarity: 0.0021
7. D8 - Public Wi-Fi and Safe Browsing | Similarity: 0.0018
8. D4 - Network Firewalls and Intrusion Detection | Similarity: 0.0018
9. D3 - Ransomware Incident Response | Similarity: 0.0000
10. D2 - Malware and Antivirus Protection | Similarity: 0.0000

Query: network firewall protection
1. D4 - Network Firewalls and Intrusion Detection | Similarity: 0.5067
2. D8 - Public Wi-Fi and Safe Browsing | Similarity: 0.1456
3. D5 - Encryption and Data Privacy | Similarity: 0.0693
4. D10 - Multi-Factor Authentication for Online Accounts | Similarity: 0.0169
5. D2 - Malware and Antiv